In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import time
import numpy as np
import pandas as pd
import dai
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ===================== 全局固定配置（训练/推理统一） =====================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# 时序窗口 64 ≤ 240 合规
SEQ_LEN = 64
# 训练超参
EPOCHS = 4
BATCH_SIZE = 256
LR = 1e-3
# 限制训练标的，控制运行时长
MAX_TRAIN_INS = 300

# 【核心】仅使用数据表原生原始字段（无人工因子）
PRICE_COLS = ["open", "high", "low", "close", "bid_price1", "ask_price1"]
VOL_COLS = ["volume", "amount", "bid_volume1", "ask_volume1"]
FEATURE_COLS = PRICE_COLS + VOL_COLS
N_FEAT = len(FEATURE_COLS)

# 模型结构配置
MODEL_CFG = {
    "n_feat": N_FEAT,
    "d_model": 64,
    "nhead": 4,
    "nlayers": 2,
    "dim_ff": 128,
    "seq_len": SEQ_LEN
}

# 训练区间（公榜固定，私榜平台重训）
TRAIN_START = "2019-01-01"
TRAIN_END = "2023-12-31 23:59:59"
MODEL_JSON_PATH = "model_weight.json"

# ===================== 模型定义 =====================
class StockTransformer(nn.Module):
    def __init__(self, n_feat, d_model=64, nhead=4, nlayers=2, dim_ff=128, seq_len=64):
        super().__init__()
        self.proj = nn.Linear(n_feat, d_model)
        self.pos_enc = nn.Parameter(torch.zeros(1, seq_len, d_model))
        # Transformer Encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=0.1,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        # 回归输出头
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        # x: [Batch, SeqLen, Feature]
        x = self.proj(x) + self.pos_enc
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.head(x).squeeze(-1)

# ===================== 权重 保存/加载（JSON纯文本，平台合规） =====================
def save_weight(ckpt, path):
    payload = {}
    for key, val in ckpt.items():
        if key == "state_dict":
            tensor_dict = {}
            for k, v in val.items():
                t = v.detach().cpu()
                tensor_dict[k] = {
                    "dtype": t.dtype.name,
                    "shape": list(t.shape),
                    "data": t.flatten().numpy().tolist()
                }
            payload["state_dict"] = tensor_dict
        else:
            payload[key] = val
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)

def load_weight(path, map_location="cpu"):
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    state_dict = {}
    for k, meta in payload["state_dict"].items():
        t = torch.tensor(meta["data"], dtype=getattr(torch, meta["dtype"]))
        state_dict[k] = t.reshape(meta["shape"]).to(map_location)
    payload["state_dict"] = state_dict
    return payload

# ===================== 工具函数 =====================
def get_csi1000_instruments(sd, ed):
    """获取区间内中证1000成分股"""
    df = dai.query(
        "SELECT DISTINCT instrument FROM bigalpha_2026_instruments",
        filters={"date": [sd, ed]},
        compression=True
    ).df()
    return df["instrument"].tolist()

def build_sequence_data(table, sd, ed, instruments, seq_len, stats=None):
    """
    构建时序样本: 仅允许 log1p、缺失值填充、标准化
    返回: 特征数组, 标签数组, 对应日期+标的, 标准化均值方差
    """
    # 前置缓冲日期，保证时序窗口充足
    buf_date = (pd.to_datetime(sd) - pd.Timedelta(days=30)).strftime("%Y-%m-%d")
    query_cols = ["date", "instrument"] + FEATURE_COLS
    sql = f"SELECT {','.join(query_cols)} FROM {table} ORDER BY instrument, date"

    df = dai.query(
        sql,
        filters={"date": [buf_date, ed], "instrument": instruments},
        compression=True
    ).df()

    # 合规预处理1：成交量类字段 log1p
    for col in VOL_COLS:
        df[col] = np.log1p(df[col].clip(lower=0))
    # 合规预处理2：缺失值填充
    df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())

    # 构造标签：单只股票下一时刻收益率（合规，非人工因子）
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df["next_close"] = df.groupby("instrument")["close"].shift(-1)
    df["label_ret"] = (df["next_close"] / df["close"] - 1).fillna(0)

    # 滑动窗口构造时序样本
    sample_x, sample_y, sample_key = [], [], []
    for ins, group in df.groupby("instrument"):
        group = group.reset_index(drop=True)
        if len(group) <= seq_len:
            continue
        feat_arr = group[FEATURE_COLS].to_numpy(dtype=np.float32)
        label_arr = group["label_ret"].to_numpy(dtype=np.float32)
        dt_arr = group["date"].to_numpy()

        for i in range(seq_len, len(group)):
            # 截取回看窗口
            win_x = feat_arr[i-seq_len:i]
            win_y = label_arr[i]
            curr_dt = dt_arr[i]
            sample_x.append(win_x)
            sample_y.append(win_y)
            sample_key.append((curr_dt, ins))

    if not sample_x:
        raise RuntimeError("无有效时序样本，请检查数据区间与股票池")

    X = np.stack(sample_x).astype(np.float32)
    y = np.array(sample_y).astype(np.float32)

    # 合规预处理3：标准化（训练集计算统计量，推理复用）
    if stats is None:
        flat_x = X.reshape(-1, N_FEAT)
        mean = flat_x.mean(axis=0)
        std = flat_x.std(axis=0) + 1e-6
        stats = (mean, std)
    mean, std = stats
    X = (X - mean) / std

    keys_df = pd.DataFrame(sample_key, columns=["date", "instrument"])
    return X, y, keys_df, stats

# ===================== 训练函数（本地执行一次生成权重） =====================
def train_and_save(table):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"训练设备: {device}")

    # 获取股票池
    ins_list = get_csi1000_instruments(TRAIN_START, TRAIN_END)[:MAX_TRAIN_INS]
    # 构建训练集
    X_train, y_train, _, train_stats = build_sequence_data(
        table, TRAIN_START, TRAIN_END, ins_list, SEQ_LEN
    )
    # 标签去极值
    lo, hi = np.percentile(y_train, [1, 99])
    y_train = np.clip(y_train, lo, hi)

    # 初始化模型
    model = StockTransformer(**MODEL_CFG).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"模型总参数量: {total_params} (10w~1亿 合规)")

    # 训练流程
    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
        batch_size=BATCH_SIZE, shuffle=True, pin_memory=(device.type == "cuda")
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_loss:.8f}")

    # 保存权重+配置+统计量
    mean, std = train_stats
    save_weight({
        "state_dict": model.state_dict(),
        "model_cfg": MODEL_CFG,
        "seq_len": SEQ_LEN,
        "mean": mean.tolist(),
        "std": std.tolist()
    }, MODEL_JSON_PATH)
    print(f"训练完成，权重已保存至: {MODEL_JSON_PATH}")

# ===================== 比赛入口 main (平台自动调用) =====================
def main(datasource, start_date, end_date):
    table_name = datasource
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 首次运行：训练并保存权重；后续直接加载
    if not os.path.exists(MODEL_JSON_PATH):
        train_and_save(table_name)

    # 加载权重与配置
    ckpt = load_weight(MODEL_JSON_PATH, map_location=device)
    model = StockTransformer(**ckpt["model_cfg"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    # 获取当前区间股票池
    curr_ins = get_csi1000_instruments(start_date, end_date)
    # 构建推理数据
    X_infer, _, key_df, _ = build_sequence_data(
        table_name, start_date, end_date, curr_ins, SEQ_LEN,
        stats=(np.array(ckpt["mean"]), np.array(ckpt["std"]))
    )

    # 模型推理
    with torch.no_grad():
        infer_tensor = torch.from_numpy(X_infer).to(device)
        scores = model(infer_tensor).cpu().numpy()

    # 拼接结果，严格输出三列
    key_df["score"] = scores
    result = key_df[["date", "instrument", "score"]].copy()
    return result

# ===================== 本地自测（可保留，不影响平台运行） =====================
if __name__ == "__main__":
    test_ds = "bigalpha_2026_stock_bar1m"
    test_start = "2023-01-01 00:00:00"
    test_end = "2023-01-15 23:59:59"
    res = main(test_ds, test_start, test_end)
    print("推理结果样例：")
    print(res.head(10))